In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

## Section 1: Standardization and Normalization Help 





In [3]:
url = 'https://gist.githubusercontent.com/tijptjik/9408623/raw/b237fa5848349a14a14e5d4107dc7897c21951f5/wine.csv'
df = pd.read_csv(url)

# the data in the wine column are the labels
labels_orig = df['Wine'].values

# the data from all other columns are the features
features_orig = df.drop(columns='Wine').values

features_orig_tensor = torch.tensor(features_orig, dtype=torch.float32)
labels_orig_tensor = torch.tensor(labels_orig, dtype=torch.long)
df

,Wine,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,3,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,3,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,3,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,3,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


Set seed

In [17]:
SEED = 2025
generator = torch.Generator().manual_seed(SEED)
generator

In [18]:
dataset = TensorDataset(features_orig_tensor, labels_orig_tensor)
train_fraction = 0.8
test_fraction = 0.2

train_data, test_data = random_split(dataset, lengths=[train_fraction, test_fraction], generator=generator)

train_dataloader = DataLoader(train_data, batch_size=20, drop_last=True)

In [19]:
len(train_data)

143

In [20]:
class Model(nn.Module):
    def __init__(self,):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(13, 16),
            nn.ReLU(),
            nn.Linear(16,13)
        )

    def forward(self, x):
        output = self.layers(x)
        return output
    
Model()

Model(
  (layers): Sequential(
    (0): Linear(in_features=13, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=13, bias=True)
  )
)

Train without scaling

In [21]:
torch.manual_seed(SEED)
model = Model()

nepochs = 100

optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

loss_function = nn.CrossEntropyLoss()

losses = []
accuracies = []
for epoch in tqdm(range(nepochs)):

    batch_losses, batch_accuracies = [], []
    for batch_idx, (features, labels) in enumerate(train_dataloader):

        optimizer.zero_grad()

        output = model.forward(features)

        loss = loss_function(output, labels)
        batch_losses.append(loss.item())

        loss.backward()

        optimizer.step()

        accuracy = (labels == torch.argmax(output, dim=1)).sum()/len(labels)
        batch_accuracies.append(accuracy)

    losses.append(np.mean(batch_losses))
    accuracies.append(np.mean(batch_accuracies))

100%|██████████| 100/100 [00:01<00:00, 68.66it/s]


Test

In [22]:
test_dataloader = DataLoader(test_data, batch_size=len(test_data))

In [23]:
with torch.no_grad():
    test_accuracies = []
    for batch_idx, (features, labels) in enumerate(test_dataloader):

        output = model(features)

        accuracy = (labels == torch.argmax(output, dim=1)).sum()/len(labels)
        test_accuracies.append(accuracy)

np.mean(test_accuracies)

np.float32(0.7714286)

With scaling

In [33]:
class Model(nn.Module):
    def __init__(self,):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(13, 16),
            nn.ReLU(),
            nn.Linear(16,13)
        )

    def forward(self, x):
        output = self.layers(x)
        return output
    
Model()

Model(
  (layers): Sequential(
    (0): Linear(in_features=13, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=13, bias=True)
  )
)

In [34]:
features_scaled_tensor = torch.tensor(StandardScaler().fit_transform(features_orig), dtype=torch.float32)

dataset = TensorDataset(features_scaled_tensor, labels_orig_tensor)
train_fraction = 0.8
test_fraction = 0.2

train_data, test_data = random_split(dataset, lengths=[train_fraction, test_fraction], generator=generator)

train_dataloader = DataLoader(train_data, batch_size=20, drop_last=True)

In [35]:
torch.manual_seed(SEED)
model = Model()

nepochs = 100

optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

loss_function = nn.CrossEntropyLoss()

losses = []
accuracies = []
for epoch in tqdm(range(nepochs)):

    batch_losses, batch_accuracies = [], []
    for batch_idx, (features, labels) in enumerate(train_dataloader):

        optimizer.zero_grad()

        output = model.forward(features)

        loss = loss_function(output, labels)
        batch_losses.append(loss.item())

        loss.backward()

        optimizer.step()

        accuracy = (labels == torch.argmax(output, dim=1)).sum()/len(labels)
        batch_accuracies.append(accuracy)

    losses.append(np.mean(batch_losses))
    accuracies.append(np.mean(batch_accuracies))

100%|██████████| 100/100 [00:01<00:00, 65.36it/s]


Test

In [36]:
test_dataloader = DataLoader(test_data, batch_size=len(test_data))

In [37]:
with torch.no_grad():
    test_accuracies = []
    for batch_idx, (features, labels) in enumerate(test_dataloader):

        output = model(features)

        accuracy = (labels == torch.argmax(output, dim=1)).sum()/len(labels)
        test_accuracies.append(accuracy)

np.mean(test_accuracies)

np.float32(0.9714286)

In [ ]:
(features_orig - torch.mean(features_orig, dim=1))/torch.std(features_orig)

## Section 2: Standardization Should be picked wisely

## Section 3: `sklearn.preprocessing` is super helpful

## Section 4: `sklearn.pipeline` wraps all this nicely